In [183]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import textwrap

In [184]:
load_dotenv()

True

In [185]:

llm = ChatGroq(
    model="llama-3.1-8b-instant",
)

In [186]:
loader = PyPDFLoader(
    file_path='../documents/Employee Referral Program.pdf'
)
pages = loader.load_and_split()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

In [187]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
)
vectorstore = Chroma.from_documents(documents=chunks, 
                                    embedding=embeddings,
                                    collection_name="referal_docs"
                                    )
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [188]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [189]:
parser = StrOutputParser()
template = """
           SYSTEM: you are a question and answer bot.
           Be concise and accurate about your response
           Respond to the following question : {question} from 
           only the provided {context} and do some
           Dont Hellucinate or provide anything from your own
           If you can not get the answer just say i dont know
           """
prompt_template = PromptTemplate.from_template(template)

In [190]:
chain = (
    {"question" : RunnablePassthrough(), "context" : retriever | format_docs}
    | prompt_template 
    | llm 
    | parser
)
response = chain.invoke("I am going to refer a employe who has 3 years of professional experience how much bonus will i get?")
print(textwrap.fill(response, width=100))

The provided information shows a referral bonus table with the following criteria:  - 8+ years of
experience: 50,000 BDT bonus - 4+ to 8 years of experience: 40,000 BDT bonus - 2+ to 4 years of
experience: 30,000 BDT bonus  Since the employee you are referring has 3 years of professional
experience, you will be eligible for a bonus based on the "2+ to 4 years" criteria.  The bonus
amount for 3 years of experience is 30,000 BDT.
